# Naive Bayes desde Cero


## ¿Cuál es el objetivo de Naive Bayes?


El objetivo de Naive Bayes es clasificar observaciones en una clase \(y\) dada una evidencia de características \(X\), usando el teorema de Bayes y asumiendo independencia condicional entre variables.

En este notebook trabajaremos con **Gaussian Naive Bayes**, ideal cuando las variables de entrada son continuas.


**Supongamos un conjunto de datos simples**

Generaremos un dataset binario sintético con 2 características para visualizar cómo se separan las clases.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets

X, y = datasets.make_classification(
    n_samples=300,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=123,
)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", alpha=0.75)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Datos sintéticos para clasificación")
plt.grid(True)
plt.show()


## ¿Cómo decide una clase "mejor"?


La clase "mejor" para un punto es aquella con mayor probabilidad posterior:

\[
P(y=c \mid x) \propto P(y=c) \prod_{j=1}^{d} P(x_j \mid y=c)
\]

- \(P(y=c)\): probabilidad previa (prior) de la clase.
- \(P(x_j \mid y=c)\): verosimilitud de cada característica para esa clase.

En Gaussian Naive Bayes, cada característica se modela con una distribución normal por clase.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

plt.figure(figsize=(7, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", alpha=0.65, label="Train")
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", edgecolor="k", alpha=0.9, label="Test")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Separación entrenamiento / prueba")
plt.grid(True)
plt.legend()
plt.show()


Una forma de evaluar qué clase es mejor para cada muestra es comparar la **log-probabilidad posterior** de cada clase y elegir la mayor.


## Demostración Matemática


### Paso 1: Teorema de Bayes

\[
P(y=c \mid x) = \frac{P(x \mid y=c)P(y=c)}{P(x)}
\]

Como \(P(x)\) es igual para todas las clases en una misma muestra, basta comparar:
\[
P(x \mid y=c)P(y=c)
\]


### Paso 2: Supuesto Naive (independencia condicional)

\[
P(x \mid y=c) = \prod_{j=1}^{d} P(x_j \mid y=c)
\]

Por lo tanto:
\[
P(y=c \mid x) \propto P(y=c)\prod_{j=1}^{d}P(x_j\mid y=c)
\]


### Paso 3: Modelo Gaussiano por característica

Para cada clase \(c\) y característica \(j\):
\[
P(x_j\mid y=c)=\frac{1}{\sqrt{2\pi\sigma_{cj}^2}}\exp\left(-\frac{(x_j-\mu_{cj})^2}{2\sigma_{cj}^2}\right)
\]

Tomamos logaritmo para mayor estabilidad numérica:
\[
\log P(y=c\mid x) = \log P(y=c) + \sum_{j=1}^{d}\log P(x_j\mid y=c) + \text{cte}
\]


### Expresión final:

Elegimos la clase:
\[
\hat{y}=\arg\max_c\left(\log P(y=c)+\sum_{j=1}^{d}\log P(x_j\mid y=c)\right)
\]


### Ejemplo numérico

Supón dos clases \(c_0\) y \(c_1\), y una muestra \(x\). Calculamos:
1. Prior de cada clase según frecuencia en entrenamiento.
2. Media y varianza de cada feature por clase.
3. Log posterior para \(c_0\) y \(c_1\).
4. Elegimos la clase con mayor valor.


## Implementación con Código


### Funciones


In [ ]:
import numpy as np


def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)


class NaiveBayes:
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self._classes = np.unique(y)
        n_classes = len(self._classes)

        # calculate mean, var, and prior for each class
        self._mean = np.zeros((n_classes, n_features), dtype=np.float64)
        self._var = np.zeros((n_classes, n_features), dtype=np.float64)
        self._priors = np.zeros(n_classes, dtype=np.float64)

        for idx, c in enumerate(self._classes):
            X_c = X[y == c]
            self._mean[idx, :] = X_c.mean(axis=0)
            self._var[idx, :] = X_c.var(axis=0)
            self._priors[idx] = X_c.shape[0] / float(n_samples)

    def predict(self, X):
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        posteriors = []

        # calculate posterior probability for each class
        for idx, c in enumerate(self._classes):
            prior = np.log(self._priors[idx])
            posterior = np.sum(np.log(self._pdf(idx, x)))
            posterior = prior + posterior
            posteriors.append(posterior)

        # return class with highest posterior probability
        return self._classes[np.argmax(posteriors)]

    def _pdf(self, class_idx, x):
        mean = self._mean[class_idx]
        var = self._var[class_idx]
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn import datasets

X, y = datasets.make_classification(
    n_samples=1000, n_features=10, n_classes=2, random_state=123
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

nb = NaiveBayes()
nb.fit(X_train, y_train)
predictions = nb.predict(X_test)

print("Naive Bayes classification accuracy", accuracy(y_test, predictions))


### Clase


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split

# Datos 2D para visualizar decisión
X_vis, y_vis = datasets.make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    n_classes=2,
    random_state=7,
)

X_train_vis, X_test_vis, y_train_vis, y_test_vis = train_test_split(
    X_vis, y_vis, test_size=0.25, random_state=7
)

# Ajustar modelo
model = NaiveBayes()
model.fit(X_train_vis, y_train_vis)
y_hat = model.predict(X_test_vis)

print("Accuracy visual demo:", accuracy(y_test_vis, y_hat))

# Mallado para contorno de clases
x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 220),
    np.linspace(y_min, y_max, 220)
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z = model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
plt.scatter(X_train_vis[:, 0], X_train_vis[:, 1], c=y_train_vis, cmap="coolwarm", alpha=0.6, label="Train")
plt.scatter(X_test_vis[:, 0], X_test_vis[:, 1], c=y_test_vis, cmap="coolwarm", edgecolor="k", label="Test")
plt.title("Naive Bayes desde Cero (frontera de decisión)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Bloque opcional tipo script (equivalente al ejemplo original)
if __name__ == "__main__":
    from sklearn.model_selection import train_test_split
    from sklearn import datasets

    X, y = datasets.make_classification(
        n_samples=1000, n_features=10, n_classes=2, random_state=123
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=123
    )

    nb = NaiveBayes()
    nb.fit(X_train, y_train)
    predictions = nb.predict(X_test)

    print("Naive Bayes classification accuracy", accuracy(y_test, predictions))
